# 01 · XDF → CSV

生の LSL 記録をセッション単位のワイド CSV に変換する.

| | |
|---|---|
| **入力** | `data/raw/<subject>/*.xdf` |
| **出力** | `data/csv/<subject>/<session>.csv` |

各 `.xdf` には同時記録された 3 ストリーム（EEG=`EmotivDataStream-EEG`, 表面筋電=`EMG_Stream`, モーションキャプチャ=`OptiTrack_BiomechIDs`）が入っている. 共通の LSL タイムスタンプで統合し, 列名はストリーム種別で接頭（`EEG_`, `EMG_`, `Markers_`）する.

> 実験データは公開していない（参加者のプライバシー保護）. このノートブックを 実行するには自前の記録を `data/raw/` 以下に配置する.

In [1]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [2]:
from motion_intent.io_xdf import summarize_xdf, xdf_to_dataframe, export_session_csv, pool_fallback_offsets

## Inspect one recording

In [4]:
xdf_files = sorted(config.RAW_DIR.rglob('*.xdf'))
assert xdf_files, f'no .xdf files under {config.RAW_DIR} - place your recordings there first'
print(f'{len(xdf_files)} xdf files under {config.RAW_DIR}')

summarize_xdf(xdf_files[0])

78 xdf files under C:\Users\rikky\research\2025\eeg-emg-motion-intention\data\raw


Stream 2: Calculated effective sampling rate 84.2808 Hz is different from specified rate 120.0000 Hz.
Stream 4: Calculated effective sampling rate 84.2778 Hz is different from specified rate 120.0000 Hz.


sub-haru_ses-1431f_task-Default_run-001_eeg.xdf: 4 streams
  EEG                      name=EmotivDataStream-EEG       shape=(10940, 37) srate=256.0000000000000
  EMG                      name=EMG_Stream                 shape=(84501, 8) srate=2000.000000000000
  Markers                  name=OptiTrack_BiomechIDs       shape=(3556, 36) srate=120.0000000000000
  RigidBodies              name=OptiTrackRigidBodies       shape=(3555, 14) srate=120.0000000000000


## Convert every session

`export_session_csv` は `<out_dir>/<拡張子違いの同名>.csv` を書き出す. 被験者フォルダ名は各 `.xdf` の親ディレクトリ名から取る.

一部の収録では、その回だけ特定ストリームのLSLクロック同期ハンドシェイクが失敗し、
`pyxdf` の自動同期 (`synchronize_clocks`) がそのストリームを諦めてしまうことがある
(症状: マージ後の `t_sec` がストリームごとに数十万秒ズレる)。同じ物理構成なら
ストリームごとのクロックオフセットは収録日を通してほぼ一定なので、被験者フォルダ内の
他の収録から測定したオフセット (`pool_fallback_offsets`) で自動的に補完する。

In [6]:
from collections import defaultdict

by_subject = defaultdict(list)
for xdf_path in xdf_files:
    by_subject[xdf_path.parent.name].append(xdf_path)

for subject, paths in by_subject.items():
    fallback_offsets = pool_fallback_offsets(paths[0].parent)
    out_dir = config.CSV_DIR / subject
    for xdf_path in paths:
        csv_path = export_session_csv(xdf_path, out_dir, fallback_offsets=fallback_offsets)
        print(csv_path.relative_to(config.DATA_DIR))

Stream 2: Calculated effective sampling rate 84.2157 Hz is different from specified rate 120.0000 Hz.
Stream 4: Calculated effective sampling rate 84.2164 Hz is different from specified rate 120.0000 Hz.


csv\haru\sub-haru_ses-1431f_task-Default_run-001_eeg.csv


Stream 3: Calculated effective sampling rate 83.6093 Hz is different from specified rate 120.0000 Hz.
Stream 4: Calculated effective sampling rate 83.6084 Hz is different from specified rate 120.0000 Hz.


csv\haru\sub-haru_ses-1431n_task-Default_run-001_eeg.csv


Stream 2: Calculated effective sampling rate 84.3518 Hz is different from specified rate 120.0000 Hz.
Stream 3: Calculated effective sampling rate 84.3517 Hz is different from specified rate 120.0000 Hz.


...

Stream 2: Calculated effective sampling rate 97.6723 Hz is different from specified rate 120.0000 Hz.
Stream 2: Calculated effective sampling rate 98.0841 Hz is different from specified rate 120.0000 Hz.


csv\lee\sub-lee_ses-15_task-Default_run-001_eeg.csv
csv\lee\sub-lee_ses-15fre_task-Default_run-001_eeg.csv


Stream 4: Calculated effective sampling rate 91.6921 Hz is different from specified rate 120.0000 Hz.
Stream 1: Calculated effective sampling rate 91.7112 Hz is different from specified rate 120.0000 Hz.
Stream 3: Calculated effective sampling rate 97.1290 Hz is different from specified rate 120.0000 Hz.
Stream 2: Calculated effective sampling rate 97.1285 Hz is different from specified rate 120.0000 Hz.
Stream 5: Calculated effective sampling rate 1519.1278 Hz is different from specified rate 2000.0000 Hz.
Stream 1: Calculated effective sampling rate 1519.5801 Hz is different from specified rate 2000.0000 Hz.


csv\lee\sub-lee_ses-176f_task-Default_run-001_eeg.csv


ValueError: columns overlap but no suffix specified: Index(['EMG_EMG0', 'EMG_EMG1', 'EMG_EMG2', 'EMG_EMG3', 'EMG_EMG4', 'EMG_EMG5',
       'EMG_EMG6', 'EMG_EMG7'],
      dtype='str')

## Quick look at the merged frame

In [7]:
if xdf_files:
    df = xdf_to_dataframe(xdf_files[0])
    print(df.shape)
    display(df.filter(regex='^(t_sec|EEG_Cz|EMG_|Markers_).*').head())

Stream 2: Calculated effective sampling rate 84.2157 Hz is different from specified rate 120.0000 Hz.
Stream 4: Calculated effective sampling rate 84.2164 Hz is different from specified rate 120.0000 Hz.


(102552, 92)


,EEG_Cz,EMG_EMG0,EMG_EMG1,EMG_EMG2,EMG_EMG3,EMG_EMG4,EMG_EMG5,EMG_EMG6,EMG_EMG7,Markers_RCAJ_X,...,Markers_LFLE_X,Markers_LFLE_Y,Markers_LFLE_Z,Markers_RFAL_X,Markers_RFAL_Y,Markers_RFAL_Z,Markers_LFAL_X,Markers_LFAL_Y,Markers_LFAL_Z,t_sec
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.491216,...,0.120688,0.435186,0.427726,0.530437,0.054141,0.615510,NaN,NaN,NaN,0.000000
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.008302
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.491219,...,0.120726,0.435181,0.427738,0.530442,0.054133,0.615512,NaN,NaN,NaN,0.011874
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.020176
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.491212,...,0.120629,0.435214,0.427700,0.530447,0.054140,0.615504,NaN,NaN,NaN,0.023749
